# Track 2 — Archive Intelligence Starter

Optional local baseline for manifest inspection and keyword search over text-like archive files. Not a finished QA system; PDF parsing and evidence-grounded answering are your next steps.

In [ ]:
from pathlib import Path
import csv, re
from html import unescape
DATA_ROOT = Path('participant/track_2_archive_intelligence')
MANIFEST = DATA_ROOT / 'metadata' / 'document_manifest.csv'
DOCUMENTS = DATA_ROOT / 'documents'
if MANIFEST.exists():
    with MANIFEST.open(newline='', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        print('Manifest columns:', reader.fieldnames)
        for _, row in zip(range(3), reader): print(row)
else: print('Set DATA_ROOT to the preloaded dataset path.')

In [ ]:
def read_local_text(path):
    raw = path.read_text(encoding='utf-8', errors='ignore')
    return unescape(re.sub(r'<[^>]+>', ' ', raw)) if path.suffix.lower() in {'.html', '.htm', '.xml'} else raw
files = [p for p in DOCUMENTS.rglob('*') if p.is_file() and p.suffix.lower() in {'.txt', '.html', '.htm', '.xml'}] if DOCUMENTS.exists() else []
print('Text-like files:', len(files))
if files: print('Preview:', read_local_text(files[0])[:500])
print('PDF documents require a local PDF parser.')

In [ ]:
def keyword_search(question, paths, top_k=3):
    terms = set(re.findall(r'\w+', question.lower())) - {'the','a','an','in','on','of','is','what','when','which'}
    results = []
    for path in paths:
        text = read_local_text(path)
        score = sum(text.lower().count(term) for term in terms)
        if score: results.append((score, str(path), text[:250]))
    return sorted(results, reverse=True)[:top_k]
print(keyword_search('well drilling report', files))

## Next steps

Add PDF parsing, evidence-based answers and uncertainty handling. Fit indexes before submission and save them. The final inference entry point must answer unseen questions without rebuilding indexes.